# ATML PA3 Task 1 — Kaggle controller

Enable GPU and Internet in Kaggle settings. Edit and run the single code cell for one phase per session. It finds a mounted `Assignment-3` root or sparse clones the course repository.

In [ ]:
from pathlib import Path
import hashlib, shutil, subprocess, sys

# Edit these controls; run this cell once per phase and session.
PHASE = "diagnose"  # diagnose, prepare, preflight, train, evaluate, weight, activations, qualitative, aggregate, export
METHOD = "full"  # full, l2sp, lora4, lora16, oplora16
CONDITION = "baseline"  # baseline, full_mid, full_final, l2sp_final, lora4_final, lora16_final, oplora16_final
STOP_UPDATE = None  # 312, 625, 937, 1250, or None
INSTALL_REQUIREMENTS = True
REPO_SOURCE = None  # optional mounted Assignment-3 root
RESTORE_DATASETS = []  # mounted previous output datasets

working = Path("/kaggle/working")
def valid_repo(path):
    return (path / "task1" / "run.py").is_file() and (path / "requirements.txt").is_file()

if REPO_SOURCE:
    repo = Path(REPO_SOURCE)
    if not valid_repo(repo):
        raise FileNotFoundError(f"Invalid Assignment-3 root: {repo}")
else:
    candidates = [working / "atml-repo" / "Assignment-3", working / "Assignment-3"]
    candidates += list(Path("/kaggle/input").glob("*/Assignment-3"))
    candidates += list(Path("/kaggle/input").glob("*"))
    repo = next((p for p in candidates if valid_repo(p)), None)
    if repo is None:
        clone = working / "atml-repo"
        if clone.exists():
            raise FileNotFoundError(f"Existing clone lacks Assignment-3/task1/run.py: {clone}")
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                        "https://github.com/Mujtaba7734/Advanced-Topics-in-Machine-Learning.git",
                        str(clone)], check=True)
        subprocess.run(["git", "-C", str(clone), "sparse-checkout", "set", "Assignment-3"], check=True)
        repo = clone / "Assignment-3"
        if not valid_repo(repo):
            raise FileNotFoundError(f"Clone lacks Assignment-3/task1/run.py: {repo}")
print("Repository:", repo, flush=True)
EXPECTED_SOURCE_HASHES = {'task1/dataset.py': '759add5fe410bfc23b39601b157516ed3661b05388b7f4e3d80b6e977479037d', 'task1/data_source.py': 'b26e41e242e5367efff231e4360c33a6f75a8c0984b2bb2311acd9d4dccbcf04', 'task1/diagnostics.py': '5b796f541273cfe217f33d768e97535b807a5ee48f2281f81c443f9866947f65', 'task1/evaluation.py': '0159b34f320b4baeb04d513d55ad92f5a5196020d4802c8f940114df688fcc56', 'task1/l2sp.py': 'fef646ed53db5544728a670bbe912d0147e384ee18f307e398149699848d13b6', 'task1/model.py': 'bd3e8a1d16972aa99c903d9b6b50dabdafaf73a3cba799223e475d72cbb30228', 'task1/optim.py': '609179daa51f8d6aa2fee35b6323a06ef2e2e494ce8a4886b3c7e984d7154a57', 'task1/plots.py': '0d25c39c461f98944a4b2ac6ecd4ba65a4a6f1f1690f808168b38d890cc8ff62', 'task1/preflight.py': '5935bd0c0a5a1c974ab5512d9b50bc8d62dd88a28c7d7e3ccb45fea2ff700192', 'task1/preflight_memory.py': 'cd5399ecbc6fa188a1c8fae77bf019de2ea267f4d2e870559cb09fb356c90056', 'task1/prepare_data.py': 'dcd071379c3072b7849b042053136f8d3e6d99fabecb980818eacddcf6c55660', 'task1/protocol.py': 'ba46e56a2771404959650d25e45720d72edf15971986e118ce8b7d2be2d979a0', 'task1/qualitative.py': '3c521312e9e94f8e6e6d8caaa04a5fe5d5eb7d197f246761f2bbabbd577d2cad', 'task1/run.py': '5706a99fca16129f5c2ebe2ff6df9cbac07e887f4d5cf8afb5a8d358da499bd3', 'task1/training.py': '057f95a5b4f6d204534b860a24c56d02580985c82ecb758892617c7369b1e10f', 'task1/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'common/checkpoint.py': '0c225b86eb5a5029b23ea22770d57e26300bdc2fdbbe4370424d27d84217d70a', 'common/device.py': 'ac0c83a24c357963b99f100b339113c723a3164119c0eac26cdc99ac5750b185', 'common/seed.py': '60e619a71d7ca690592ece3669fc87dd399495734126c2aadf68fcec3da306a1', 'common/__init__.py': 'e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855', 'configs/task1.yaml': '0710d223eb40bf6579405ed251103dfe1ec0790f5bd1fcb6298b39e9c667851b', 'requirements.txt': '1cd65cbeb944cb2392c0b78fdb3d6fdb853d1c00bec40073732ff36ac604772a', 'kaggle/build_notebook.py': 'edc01705b515a618c3d6e75c26094fd1b401377a6c206ea169a16d8c2b63a2d7'}
for relative, expected in EXPECTED_SOURCE_HASHES.items():
    source = repo / relative
    if not source.is_file() or hashlib.sha256(source.read_bytes()).hexdigest() != expected:
        raise RuntimeError(f'Audited code mismatch at {source}. Upload the current Assignment-3 folder as a Kaggle input.')

def copy_verified(source, target):
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        digest = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
        if digest(source) != digest(target):
            raise ValueError(f"Conflicting restored file: {target}. Use one current output dataset.")
        return
    shutil.copy2(source, target)

for mount in map(Path, RESTORE_DATASETS):
    if not mount.is_dir():
        raise FileNotFoundError(mount)
    for relative in ("outputs/task1/data", "outputs/task1/preflight", "results/task1"):
        source_dir = mount / relative
        if source_dir.exists():
            for source in source_dir.rglob("*"):
                if source.is_file():
                    copy_verified(source, working / relative / source.relative_to(source_dir))
    methods = [METHOD] if PHASE in ("train", "export") else ["full", "l2sp", "lora4", "lora16", "oplora16"]
    for method in methods:
        source_method = mount / "outputs/task1" / method
        if not source_method.exists():
            continue
        recovery = sorted(source_method.glob("recovery_*.pt"))
        for source in source_method.iterdir():
            if not source.is_file() or (source.name.startswith("recovery_") and PHASE != "export"):
                continue
            if PHASE not in ("train", "export") and source.name == "latest.pt":
                continue
            copy_verified(source, working / "outputs/task1" / method / source.name)
        if PHASE == "train" and not (working / "outputs/task1" / method / "latest.pt").exists() and recovery:
            copy_verified(recovery[-1], working / "outputs/task1" / method / recovery[-1].name)
    print("Restored:", mount, flush=True)

if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", str(repo / "requirements.txt")], check=True)

import yaml
config = yaml.safe_load((repo / "configs/task1.yaml").read_text(encoding="utf-8"))
prompt_path = repo / "first_plot_questions.yaml"
if not prompt_path.is_file():
    matches = list(Path("/kaggle/input").glob("*/first_plot_questions.yaml"))
    matches += list(Path("/kaggle/input").glob("*/Assignment-3/first_plot_questions.yaml"))
    if len(matches) != 1:
        raise FileNotFoundError("Mount first_plot_questions.yaml, or include it in the repository")
    prompt_path = matches[0]
config["qualitative_prompts"] = str(prompt_path)
runtime_config = working / "task1_runtime.yaml"
runtime_config.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
print("Prompt file:", prompt_path, flush=True)

cmd = [sys.executable, "-m", "task1.run", PHASE, "--config", str(runtime_config)]
if PHASE in ("train", "export"):
    cmd += ["--method", METHOD]
if PHASE == "train" and STOP_UPDATE is not None:
    cmd += ["--stop-update", str(STOP_UPDATE)]
if PHASE in ("evaluate", "qualitative"):
    cmd += ["--condition", CONDITION]
print("Running:", " ".join(cmd), flush=True)
subprocess.run(cmd, cwd=repo, check=True)


## Phase order

1. `diagnose`, `prepare`, then **`preflight`**. Inspect `outputs/task1/preflight/report.json`; training requires its pass marker. Existing prepared data is validated and reused.
2. Evaluate `baseline`; train each method, resuming at checkpoints 312, 625, 937, 1250 as needed.
3. Evaluate the six trained conditions; run `weight`, `activations`, and `qualitative` for all seven conditions.
4. Run `aggregate`, review results, then export and save outputs.

Save `/kaggle/working/outputs/task1` and `/kaggle/working/results/task1` before a session ends. Mount the saved dataset and set `RESTORE_DATASETS` in the next session. Full FT and L2-SP put model computation on GPU 0 and AdamW master weights and moments on GPU 1. OPLoRA uses GPU 1 for FP32 SVD workspace. The two T4s have separate VRAM, and actual fit remains a preflight requirement.